In [25]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, classification_report, confusion_matrix,\
accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

import lightgbm as lgb
import joblib
import re

In [4]:
X_train, y_train = pd.read_csv('X_train.csv'), pd.read_csv('y_train.csv')
X_test = pd.read_csv('X_test.csv')

In [6]:
X_train.head()

,Дата,Группа,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Упоминание брендов,Уровень платежеспособности,Возраст,Заголовок,Текст
0,19.03.2024,Аккумуляторы Машина,M-9591367848,текстовый,без изображения,мобильные,мужской,Целевые запросы,Без упоминания вашего бренда,Остальные,45-54,Выберите авто аккумулятор с выгодой,Аккумуляторы для легковую машину в Иркутске. Д...
1,18.09.2023,Аккумуляторы Новый,M-9591367816,текстовый,без изображения,мобильные,мужской,не определено,не определено,6-10%,25-34,Новые Аккумуляторы,Centra Market - Большой выбор новых аккумулято...
2,07.02.2024,Аккумуляторы Иркутск,M-9591367718,текстовый,без изображения,мобильные,мужской,Широкие запросы,не определено,Остальные,старше 55,Большой выбор Авто аккумуляторов,Гарантия. Сервис. Свяжитесь со специалистом ил...
3,08.10.2023,Аккумуляторы Купить Цена,M-9591367860,графический,с изображением,десктоп,женский,Целевые запросы,не определено,6-10%,35-44,Купить аккумулятор с выгодой,Выгодные цены на аккумуляторы в Иркутске. Дост...
4,05.07.2023,Аккумуляторы Магазин,M-9591367826,текстовый,без изображения,мобильные,женский,Целевые запросы,не определено,Остальные,35-44,Большой магазин Авто аккумуляторов,Свяжитесь со специалистом или выберите аккумул...


In [8]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63445 entries, 0 to 63444
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Дата                        63445 non-null  object
 1   Группа                      63445 non-null  object
 2   № Объявления                63445 non-null  object
 3   Формат                      63445 non-null  object
 4   Размер изображения          63445 non-null  object
 5   Тип устройства              63445 non-null  object
 6   Пол                         63445 non-null  object
 7   Категория таргетинга        63445 non-null  object
 8   Упоминание брендов          63445 non-null  object
 9   Уровень платежеспособности  63445 non-null  object
 10  Возраст                     63445 non-null  object
 11  Заголовок                   63445 non-null  object
 12  Текст                       63445 non-null  object
dtypes: object(13)
memory usage: 6.3+ MB


In [5]:
y_train = y_train > 0
y_train = y_train.astype(int)
y_train = y_train['0']

In [27]:
# ------------------------------------------------------------
# 1) Подготовка данных: предположение — X_train, y_train уже доступны
# Если у вас есть столбец wCTR, то:
# y = (df['wCTR'] > 0).astype(int)
# ------------------------------------------------------------

# Для примера: используйте X_train, y_train как есть
# X_train: DataFrame с колонками, описанными в вопросе
# y_train: Series (0/1)  — если у вас wCTR, сначала преобразуйте

# ------------------------------------------------------------
# 2) Функции для извлечения простых текстовых признаков
# ------------------------------------------------------------
class TextStatsExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, text_cols):
        self.text_cols = text_cols

    def fit(self, X, y=None):
        return self

    def _stats_for_series(self, s):
        s = s.fillna('')
        return pd.DataFrame({
            f'{col}_len': s.apply(len) for col in [0]  # placeholder, will replace below
        })

    def transform(self, X):
        # X — DataFrame or numpy array with text columns concatenated if needed
        # We'll expect X to be a DataFrame with the columns provided in self.text_cols
        df = pd.DataFrame()
        for col in self.text_cols:
            s = X[col].fillna('').astype(str)
            df[f'{col}_len'] = s.str.len()
            df[f'{col}_words'] = s.str.split().apply(len)
            df[f'{col}_digits'] = s.str.count(r'\d')
            df[f'{col}_exclam'] = s.str.count(r'!')
            df[f'{col}_upper_ratio'] = s.apply(lambda t: (sum(1 for ch in t if ch.isupper()) / max(len(t), 1)))
        return df

# ------------------------------------------------------------
# 3) Предобработка даты
# ------------------------------------------------------------
def extract_date_features(df, date_col='Дата'):
    s = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    res = pd.DataFrame({
        'date_day': s.dt.day.fillna(-1).astype(int),
        'date_month': s.dt.month.fillna(-1).astype(int),
        'date_year': s.dt.year.fillna(-1).astype(int),
        'date_weekday': s.dt.weekday.fillna(-1).astype(int),
        'date_is_weekend': s.dt.weekday.isin([5,6]).astype(int).fillna(0)
    })
    return res

# ------------------------------------------------------------
# 4) Pipeline для текстовых данных: TF-IDF + SVD (для уменьшения размерности)
# ------------------------------------------------------------
# Объединяем Заголовок и Текст в один для TF-IDF (или можно отдельно)
def concat_title_text(df, title_col='Заголовок', text_col='Текст'):
    return (df[title_col].fillna('').astype(str) + ' ' + df[text_col].fillna('').astype(str)).values

# TF-IDF vectorizer (на объединённом тексте)
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)

# SVD для сжатия
svd = TruncatedSVD(n_components=128, random_state=42)

# ------------------------------------------------------------
# 5) Список фичей
# ------------------------------------------------------------
date_col = 'Дата'
text_cols = ['Заголовок', 'Текст']
cat_cols = ['Группа', 'Формат', 'Размер изображения', 'Тип устройства',
            'Пол', 'Категория таргетинга', 'Упоминание брендов',
            'Уровень платежеспособности', 'Возраст']
# Исключаем '№ Объявления' — скорее ID

# ------------------------------------------------------------
# 6) Собираем фичи вручную, затем передадим в модель LightGBM
# ------------------------------------------------------------
def build_feature_matrix(X):
    # X — DataFrame
    # 1) date features
    date_feats = extract_date_features(X, date_col=date_col)

    # 2) text stats
    tse = TextStatsExtractor(text_cols=text_cols)
    text_stats = tse.transform(X)

    # 3) categorical (fillna)
    cat_df = X[cat_cols].fillna('NA').astype(str)

    # 4) tfidf + svd on concatenated text
    combined_text = concat_title_text(X, title_col='Заголовок', text_col='Текст')
    tfidf_mat = tfidf.fit_transform(combined_text)  # если позже — использовать transform
    svd_mat = svd.fit_transform(tfidf_mat)

    # 5) объединяем всё в DataFrame (svd матрица в колонки)
    svd_df = pd.DataFrame(svd_mat, index=X.index, columns=[f'svd_{i}' for i in range(svd_mat.shape[1])])
    features = pd.concat([date_feats.reset_index(drop=True), text_stats.reset_index(drop=True),
                          cat_df.reset_index(drop=True), svd_df.reset_index(drop=True)], axis=1)
    return features

# ------------------------------------------------------------
# Подбор гиперпараметров LightGBM
# ------------------------------------------------------------
def tune_lgbm_params(features_train, y_train):
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    # пространство поиска
    param_dist = {
        'num_leaves': randint(15, 80),
        'learning_rate': uniform(0.01, 0.2),
        'max_depth': randint(3, 12),
        'min_child_samples': randint(5, 50),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3),
        'reg_alpha': uniform(0, 1),
        'reg_lambda': uniform(0, 1),
    }

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=30,            # можно увеличить до 50–100
        scoring='roc_auc',
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    search.fit(features_train, y_train)

    print("Лучшие параметры:", search.best_params_)
    print("Лучший ROC AUC:", search.best_score_)

    return search.best_params_


# ------------------------------------------------------------
# 7) Модель: LightGBM (sklearn API)
# ------------------------------------------------------------
def train_lightgbm(X, y, fit_feature_builder=True):
    # Разделим часть для валидации
    X_train_part, X_val, y_train_part, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Если нужно — строим признаки (fit tfidf, svd на тренировочной части)
    # Для простоты: будем строить признаки на всей X_train_part, затем применять transform на X_val
    # Реализация: fit tfidf+svd на X_train_part, затем transform обеих частей
    # Чтобы не усложнять — применяем build_feature_matrix (включает fit tfidf, svd)
    # В реальном пайплайне: отдельно fit на train и transform на val/test.
    features_train = build_feature_matrix(X_train_part)
    # нужно сохранить tfidf и svd, поэтому для val делаем transform using fitted ones
    # For simplicity in this script we rebuild them globally (not perfect) — production: separate fit/transform.

    for col in cat_cols:
        if col in features_train.columns:
            features_train[col] = features_train[col].astype('category')

    # подобрать гиперпараметры
    best_params = tune_lgbm_params(features_train, y_train_part)
    
    print("Используем параметры после поиска:", best_params)


    # Fit LightGBM
    clf = lgb.LGBMClassifier(
        objective='binary',
        random_state=42,
        n_jobs=-1,
        **best_params
    )


    # compute scale_pos_weight to handle imbalance
    pos = sum(y_train_part == 1)
    neg = sum(y_train_part == 0)
    if pos > 0:
        clf.set_params(scale_pos_weight=(neg/pos))

    for col in cat_cols:
        features_train[col] = features_train[col].astype('category')

    clf.fit(features_train, y_train_part, eval_set=[(features_train, y_train_part)])

    # For validation, we need to produce features val: to keep script consistent, rebuild features on X_val (not ideal but acceptable)
    features_val = build_feature_matrix(X_val)  # NOTE: in real pipeline: reuse fitted tfidf/svd to transform

    for col in cat_cols:
        features_val[col] = features_val[col].astype('category')

    y_pred_proba = clf.predict_proba(features_val)[:,1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    print("ROC AUC:", roc_auc_score(y_val, y_pred_proba))
    print("PR AUC (average precision):", average_precision_score(y_val, y_pred_proba))
    print("Accuracy:", accuracy_score(y_val, y_pred))
    print("Classification report:")
    print(classification_report(y_val, y_pred, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_pred))
    return clf

# ------------------------------------------------------------
# 8) Запуск (пример)
# ------------------------------------------------------------

clf = train_lightgbm(X_train, y_train)

# ------------------------------------------------------------
# 9) Сохранение модели
# ------------------------------------------------------------
# joblib.dump(clf, 'lgbm_acc_binaries_model.joblib')

# ------------------------------------------------------------
# ПРИМЕЧАНИЯ (важно)
# - В production пайплайне: разделяйте fit/transform для TF-IDF и SVD (fit на train, transform на val/test).
# - Можно ускорить: использовать HashingVectorizer + TruncatedSVD без fit TF-IDF (stateless).
# - Для кат. признаков LightGBM хорошо работает с LabelEncoder/категорическим dtype (и параметром categorical_feature).
# - Для проверок используйте StratifiedKFold и RandomizedSearchCV по гиперпараметрам.
# - Для объяснимости используйте SHAP (shap.TreeExplainer).
# ------------------------------------------------------------


Fitting 3 folds for each of 30 candidates, totalling 90 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 4965, number of negative: 28872
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.630803 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8646
[LightGBM] [Info] Number of data points in the train set: 33837, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.146733 -> initscore=-1.760459
[LightGBM] [Info] Start training from score -1.760459
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

Exception ignored on calling ctypes callback function: <function _log_callback at 0x7741d9c2fb50>
Traceback (most recent call last):
  File "/home/kirill/miniconda3/envs/DScource/lib/python3.10/site-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
KeyboardInterrupt: 


KeyboardInterrupt: 

In [23]:
features_test = build_feature_matrix(X_test)

for col in cat_cols:
    features_test[col] = features_test[col].astype('category')

y_test_proba = clf.predict_proba(features_test)[:, 1]

y_test_pred = (y_test_proba >= 0.5).astype(int)
pd.DataFrame(y_test_pred, columns=['0']).to_csv('submission.csv', index=False)